# Plantilla de algoritmo. Parte 3 (Problema del Viajante)

**Cómo usarla:** copia este archivo como `parte3/<algoritmo>_<tunombre>.ipynb` (por ejemplo, `parte3/fuerza_bruta_pablo.ipynb`) y trabaja solo **debajo de la línea ✂️**. Lo de arriba (preparación) es el mismo código de la Parte 2 del trabajo y sirve para cargar las versiones del grafo: **no lo modifiques**. La guía completa está en `CONTRIBUTING.md`.

> ⚙️ El notebook se ejecuta desde su carpeta (`parte3/`): la primera celda sube a la raíz del proyecto para que las rutas (`data/...`) funcionen.

In [1]:
# Sube a la raíz del proyecto para que las rutas (data/ciudades.csv, data/versiones/...) funcionen
import os
from pathlib import Path
if Path.cwd().name == "parte3":
    os.chdir("..")

In [2]:
# Librerías que se usan en todo el notebook
import csv                       # leer y escribir tablas en formato CSV
import itertools                 # recorrer todas las combinaciones de parámetros
import unicodedata               # quitar acentos de los nombres (Cataluña -> cataluna)
from pathlib import Path         # rutas de archivos

import networkx as nx            # crear y analizar grafos (la misma librería de las prácticas)
import numpy as np               # tablas de números (las tablas de distancias)
from scipy.sparse import csr_matrix                 # matriz del grafo para el algoritmo de Dijkstra
from scipy.sparse.csgraph import dijkstra           # caminos mínimos desde varios orígenes
from scipy.spatial import cKDTree                   # buscar rápido el nodo más cercano a un punto

from tsp_utils import dibujar_red  # función de dibujo de la red de carreteras, comentada en tsp_utils.py

In [3]:
# ── Parámetros de las versiones (editables) ──
X_KM  = 10             # una ciudad a más de X_KM km de la red se deja fuera de la versión
N_MIN = 4              # mínimo de ciudades para una versión por comunidad autónoma
RECALCULAR = False     # True: vuelve a calcular las 16 tablas (necesita el grafo G de la Parte 1)
CARPETA = Path("data/versiones")     # aquí se guardan las tablas ya calculadas

# Qué tipos de vía (`highway`) puede usar cada versión de carreteras
CARRETERAS = {
    "completo":             ["motorway", "motorway_link", "trunk", "trunk_link",
                             "primary", "primary_link", "secondary", "secondary_link"],
    "autovias_principales": ["motorway", "motorway_link", "trunk", "trunk_link",
                             "primary", "primary_link"],
    "solo_autovias":        ["motorway", "motorway_link"],
    "sin_autovias":         ["trunk", "trunk_link", "primary", "primary_link",
                             "secondary", "secondary_link"],
}
PESOS = {"km": ("length", 1000), "tiempo": ("travel_time", 1)}   # peso -> (atributo de la arista, divisor)
CONJUNTOS = ["capitales", "capitales_y_100k"]                    # qué ciudades se visitan
GENERALES = ["multigrafo_dirigido", "grafo_dirigido",
             "multigrafo_no_dirigido", "grafo_simple_no_dirigido"]

In [4]:
# Lee la lista de ciudades (nombre, comunidad, si es capital de provincia, coordenadas...)
with open("data/ciudades.csv", encoding="utf-8") as f:
    CIUDADES = list(csv.DictReader(f))
for c in CIUDADES:
    c["lat"], c["lon"] = float(c["lat"]), float(c["lon"])
    c["capital"] = c["capital_provincia"] == "True"

def slug(texto):
    # "Castilla y León" -> "castilla_y_leon" (sin acentos ni espacios, para usarlo como nombre)
    sin_acentos = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return sin_acentos.lower().replace(" ", "_").replace("-", "_")

AMBITOS = ["espana"] + sorted({slug(c["comunidad"]) for c in CIUDADES})

def ciudades_de(nodos, ambito):
    # Nombres de las ciudades de un conjunto: nodos = "capitales" o "capitales_y_100k";
    # ambito = "espana" o una comunidad (p. ej. "galicia")
    return [c["nombre"] for c in CIUDADES
            if (c["capital"] or nodos == "capitales_y_100k")
            and (ambito == "espana" or slug(c["comunidad"]) == ambito)]

print(f"{len(CIUDADES)} ciudades | {len(ciudades_de('capitales', 'espana'))} capitales | ámbitos: {len(AMBITOS) - 1} comunidades")
print("Capitales de Galicia:", ciudades_de("capitales", "galicia"))

75 ciudades | 47 capitales | ámbitos: 15 comunidades
Capitales de Galicia: ['A Coruña', 'Ourense', 'Lugo', 'Pontevedra']


In [5]:
def nombre_tabla(aristas, peso, dirigido):
    return CARPETA / f"{aristas}__{peso}__{'dirigido' if dirigido else 'no_dirigido'}.csv"

def guardar_tabla(D, nombres, ruta):
    with open(ruta, "w", encoding="utf-8") as f:
        np.savetxt(f, D, delimiter=",", header=",".join(nombres), comments="", fmt="%.3f")

def leer_tabla(aristas, peso, dirigido):
    ruta = nombre_tabla(aristas, peso, dirigido)
    with open(ruta, encoding="utf-8") as f:
        nombres = f.readline().strip().split(",")        # primera fila: nombres de las ciudades
    return np.loadtxt(ruta, delimiter=",", skiprows=1, encoding="utf-8"), nombres

def escribir_csv(ruta, filas, campos):
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campos)
        w.writeheader()
        w.writerows(filas)

In [6]:
UNIDAD = {"km": "km", "tiempo": "segundos"}

def cargar_version(aristas, nodos, ambito, peso, general):
    # aristas: "completo" | "autovias_principales" | "solo_autovias" | "sin_autovias"
    # nodos:   "capitales" | "capitales_y_100k"       ambito: "espana" | "galicia" | "andalucia" ...
    # peso:    "km" | "tiempo"                        general: uno de GENERALES
    # (multigrafo y grafo simple dan el mismo grafo de ciudades: no hay aristas repetidas)
    dirigido = "no_dirigido" not in general
    D, nombres = leer_tabla(aristas, peso, dirigido)
    elegidas = [c for c in ciudades_de(nodos, ambito) if c in nombres]    # sin las que quedaron fuera
    if ambito != "espana" and len(elegidas) < N_MIN:
        raise ValueError(f"Solo hay {len(elegidas)} ciudades (mínimo {N_MIN}) en {ambito}")
    K = nx.DiGraph() if dirigido else nx.Graph()
    for c in CIUDADES:
        if c["nombre"] in elegidas:
            K.add_node(c["nombre"], lat=c["lat"], lon=c["lon"])
    for p, a in enumerate(elegidas):
        for q, b in enumerate(elegidas):
            if p != q and (dirigido or p < q):                            # no dirigido: una arista por par
                K.add_edge(a, b, peso=float(D[nombres.index(a), nombres.index(b)]))
    K.graph.update(aristas=aristas, nodos=nodos, ambito=ambito, peso=peso,
                   unidad=UNIDAD[peso], general=general,
                   fuera=[c for c in ciudades_de(nodos, ambito) if c not in nombres])
    return K

In [7]:
# [AYUDA COMPARTIDA] Funciones comunes a todos los algoritmos
import time
import tracemalloc

def coste_ruta(K, ruta):
    # Coste total del ciclo: suma de las distancias entre ciudades consecutivas y de la última a la primera
    return sum(K[ruta[i]][ruta[(i + 1) % len(ruta)]]["peso"] for i in range(len(ruta)))

def validar_ruta(K, ruta):
    # Comprueba que la ruta visita cada ciudad de K exactamente una vez
    assert len(ruta) == K.number_of_nodes() and set(ruta) == set(K.nodes), "la ruta no visita cada ciudad una vez"

def medir(algoritmo, K, memoria=False, **opciones):
    # Ejecuta algoritmo(K, **opciones), comprueba el resultado y devuelve cuánto ha tardado
    # (con memoria=True mide también el pico de memoria en una segunda ejecución, porque medirla ralentiza)
    t0 = time.perf_counter()
    ruta, coste = algoritmo(K, **opciones)
    segundos = time.perf_counter() - t0
    validar_ruta(K, ruta)
    assert abs(coste - coste_ruta(K, ruta)) < 1e-6 * max(1.0, abs(coste)), "el coste devuelto no coincide con el de la ruta"
    resultado = {"algoritmo": algoritmo.__name__, "ciudades": K.number_of_nodes(), "coste": round(coste, 3),
                 "unidad": K.graph.get("unidad"), "segundos": round(segundos, 4)}
    if memoria:
        tracemalloc.start()
        algoritmo(K, **opciones)
        _, pico = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        resultado["memoria_mb"] = round(pico / 1e6, 2)
    return resultado

---
### ✂️ A PARTIR DE AQUÍ SE INTEGRA EN EL TRABAJO

Todo lo que escribas **debajo de esta línea** pasará al notebook principal (`trabajo_tsp.ipynb`, Parte 3). No modifiques esta celda.

### Recocido simulado (MCMC)

**🗂️ FICHA**

| | |
|---|---|
| **Autor** | Juan |
| **Función** | `enfriamiento_simulado(K, inicio=None)` |
| **Tipo** | Metaheurística |
| **Determinista** | No (estocástico) |
| **Garantía de calidad** | Ninguna |
| **Distancias simétricas** | No la necesita |
| **Complejidad en tiempo** | O(niveles de temperatura × iteraciones por nivel) |
| **Complejidad en memoria** | O(n²) (caché de pesos) |
| **Límite práctico** | Todas |
| **Uso de IA** | Pendiente de indicar por el autor. |

<div style="border-left:5px solid #4C72B0;background:#EEF2FA;padding:10px 14px;margin:10px 0;border-radius:4px;">

**📚 TEORÍA**

**1. Orígenes**
El **Enfriamiento Simulado** (*Simulated Annealing*, propuesto por S. Kirkpatrick en 1983) es una metaheurística de optimización inspirada en el proceso físico de recocido en metalurgia. Aunque adopta una terminología termodinámica (energía, temperatura, enfriamiento), el algoritmo es matemáticamente una aplicación directa de los métodos **MCMC (Markov Chain Monte Carlo)**.

Concretamente, implementa el algoritmo de **Metropolis-Hastings** desarrollado en 1953 para simular sistemas en equilibrio térmico. Mientras que MCMC se usa genéricamente para muestrear distribuciones de probabilidad, el Enfriamiento Simulado orienta este motor estocástico hacia la optimización global, reduciendo gradualmente el parámetro $T$ (Temperatura) para colapsar la distribución de probabilidad exclusivamente sobre los estados de mínima energía (el camino más corto).

**2. Funcionamiento del Algoritmo**
El problema fundamental de las heurísticas tradicionales (como el 2-opt voraz o el vecino más cercano) es que se atascan en **óptimos locales**: valles subóptimos de los que no pueden salir porque cualquier cambio inmediato empeora la ruta.

El Enfriamiento Simulado supera esta barrera imitando la termodinámica:
1. **Generación de Sucesores:** A partir de un estado `actual`, se genera una ruta `candidata` aplicando un operador de vecindad (en nuestro caso, cortando y uniendo dos calles, equivalente geométrico a 2-opt).
2. **Criterio de Aceptación (Física Estadística):** Si el cambio reduce los kilómetros ($\Delta F < 0$), se acepta siempre. Si empeora la ruta, no se descarta automáticamente; se acepta con una probabilidad dada por la distribución de Boltzmann: $P = e^{-\Delta F / T}$.
3. **Enfriamiento:** A temperaturas altas, el algoritmo acepta muchos errores, lo que le permite "saltar" fuera de los cráteres subóptimos (diversificación). A medida que $T$ disminuye iterativamente mediante un factor $\alpha$, el sistema se cristaliza, volviéndose cada vez más voraz hasta alcanzar la convergencia en el fondo del valle (intensificación).

**3. Modificaciones Arquitectónicas**

Este código trasciende la implementación académica estándar al integrar mejoras algorítmicas avanzadas:
* **Filtro Termodinámico $\mathcal{O}(1)$:** En lugar de calcular el coste total $\mathcal{O}(N)$ de cada permutación generada, se pre-evaluá el salto probabilístico midiendo de forma instantánea únicamente la variación de las cuatro aristas directamente afectadas ($\Delta F_{est}$), permitiendo instanciar la memoria de la nueva ruta solo cuando la termodinámica del sistema valida el cambio.
* **Caché en Memoria:** Se extrae de forma íntegra la topología del grafo geográfico hacia un diccionario hash estricto en memoria, lo que suprime por completo la pesada sobrecarga de consultas y llamadas a métodos internos de la librería NetworkX, logrando accesos de lectura sumamente rápidos.
* **Cola de Enfriamiento Dinámica:** El número de iteraciones de la cadena escala de manera fluida y adaptativa a medida que desciende la temperatura del sistema, asegurando una fase inicial de amplia diversificación espacial y concentrando la máxima densidad de exploración topológica en las etapas finales del proceso.



... 

---
**Complejidad.**
* **Espacial:** $\mathcal{O}(V + E)$ para instanciar el mapa de pesos y mantener un número constante de vectores de ciudades de tamaño $N$.
* **Temporal:** Gracias al filtro matemático local, la complejidad por iteración intentada es $\mathcal{O}(1)$. La complejidad total está acotada por $K \cdot I$, donde $K$ es el número de descensos de temperatura y $I$ las iteraciones por nivel, y en las pruebas de abajo resuelve 47 ciudades en ≈ 1 s y 75 en 2–3 s.

**Comparativa de Rendimiento**
* **Frente a Exactos (DFJ / Fuerza Bruta):** Fuerza bruta y Held–Karp sufren la explosión combinatoria (no pasan de ≈ 11 y ≈ 20 ciudades); DFJ, con un solver de programación entera, llega a 75 ciudades en unos 3 s, pero con garantía de óptimo. El recocido simulado escala de forma casi lineal en el tiempo de respuesta, aunque sin garantía.
* **Frente a Voraces:** Destruye el error provocado por métodos "miopes", mejorando claramente sus rutas. Respecto a la cota inferior del 1-Árbol, el error máximo medido en las pruebas de abajo va del 8 % al 30 % (la cota es floja, así que el error real es menor).

**Propiedades.**

- **Tipo:** metaheurística estocástica (no es exacta).
- **¿Siempre encuentra el óptimo?** No, aunque en alguna de las pruebas lo alcanza (p. ej. 5.982 km con 47 capitales, +1,1 % sobre el óptimo de 5.917,6 km).
- **¿Determinista?** No: usa azar (`random`). En el notebook se fija `random.seed(42)` antes de las pruebas para que los resultados se puedan repetir.
- **¿Garantía de calidad?** Ninguna. Solo se puede acotar el error con una cota inferior (el 1-Árbol).
- **¿Necesita distancias simétricas?** No: usa el coste de cada arista en su sentido.
- **¿Hasta cuántas ciudades?** Todas las del trabajo (47 en ≈ 1 s y 75 en 2–3 s).
</div>

**PSEUDOCÓDIGO**

<pre style="background-color: #1e1e1e; color: #d4d4d4; padding: 20px; border-radius: 8px; font-family: Consolas, 'Courier New', monospace; font-size: 14.5px; line-height: 1.6; overflow-x: auto; border: 1px solid #444;">
<b style="color: #569cd6; font-size: 16px;">enfriamiento_simulado</b>(<span style="color: #9cdcfe;">K</span>, <span style="color: #9cdcfe;">inicio</span>, <span style="color: #9cdcfe;">α</span>, <span style="color: #9cdcfe;">T_min</span>)
1. <span style="color: #9cdcfe;">W</span> ← <span style="color: #dcdcaa;">inicializar_cache_pesos</span>(<span style="color: #9cdcfe;">K</span>)                                       
   <span style="color: #9cdcfe;">N</span> ← cantidad de nodos en <span style="color: #9cdcfe;">K</span>                                            
   <span style="color: #9cdcfe;">actual</span> ← <span style="color: #dcdcaa;">genera_estado_inicial</span>(<span style="color: #9cdcfe;">N</span>)                                    
   <span style="color: #9cdcfe;">valor_actual</span> ← <span style="color: #dcdcaa;">valoracion</span>(<span style="color: #9cdcfe;">actual</span>, <span style="color: #9cdcfe;">W</span>)                                  
   <span style="color: #9cdcfe;">mejor</span> ← <span style="color: #9cdcfe;">actual</span>                                                       
   <span style="color: #9cdcfe;">mejor_valor</span> ← <span style="color: #9cdcfe;">valor_actual</span>                                             
   <span style="color: #9cdcfe;">T</span> ← <span style="color: #9cdcfe;">valor_actual</span> * <span style="color: #b5cea8;">0.15</span>                                                
   <span style="color: #9cdcfe;">n_iter</span> ← <span style="color: #dcdcaa;">max</span>(<span style="color: #b5cea8;">200</span>, <span style="color: #9cdcfe;">N</span> * <span style="color: #b5cea8;">25</span>)                                                

2. <b style="color: #c586c0;">Mientras</b> <span style="color: #9cdcfe;">T</span> &gt; <span style="color: #9cdcfe;">T_min</span>:
   2.1 <b style="color: #c586c0;">Repetir</b> <span style="color: #9cdcfe;">n_iter</span> veces:
       2.1.1 <span style="color: #9cdcfe;">i</span>, <span style="color: #9cdcfe;">j</span> ← <span style="color: #dcdcaa;">generar_indices</span>(<span style="color: #b5cea8;">0</span>, <span style="color: #9cdcfe;">N</span>-<span style="color: #b5cea8;">1</span>) con <span style="color: #9cdcfe;">i</span> &lt; <span style="color: #9cdcfe;">j</span>                      
             <span style="color: #9cdcfe;">ΔF_est</span> ← <span style="color: #9cdcfe;">W</span>(<span style="color: #9cdcfe;">actual</span>[<span style="color: #9cdcfe;">i</span>-<span style="color: #b5cea8;">1</span>], <span style="color: #9cdcfe;">actual</span>[<span style="color: #9cdcfe;">j</span>]) + <span style="color: #9cdcfe;">W</span>(<span style="color: #9cdcfe;">actual</span>[<span style="color: #9cdcfe;">i</span>], <span style="color: #9cdcfe;">actual</span>[(<span style="color: #9cdcfe;">j</span>+<span style="color: #b5cea8;">1</span>)<b style="color: #c586c0;">mod</b><span style="color: #9cdcfe;">N</span>]) - <span style="color: #9cdcfe;">W</span>(<span style="color: #9cdcfe;">actual</span>[<span style="color: #9cdcfe;">i</span>-<span style="color: #b5cea8;">1</span>], <span style="color: #9cdcfe;">actual</span>[<span style="color: #9cdcfe;">i</span>]) - <span style="color: #9cdcfe;">W</span>(<span style="color: #9cdcfe;">actual</span>[<span style="color: #9cdcfe;">j</span>], <span style="color: #9cdcfe;">actual</span>[(<span style="color: #9cdcfe;">j</span>+<span style="color: #b5cea8;">1</span>)<b style="color: #c586c0;">mod</b><span style="color: #9cdcfe;">N</span>])
       2.1.2 <b style="color: #c586c0;">Si</b> <span style="color: #9cdcfe;">ΔF_est</span> &lt; <span style="color: #b5cea8;">0</span> <b style="color: #c586c0;">ó</b>
                <b style="color: #c586c0;">se acepta</b> (con probabilidad de aceptación <span style="color: #dcdcaa;">e</span><sup>-<span style="color: #9cdcfe;">ΔF_est</span>/<span style="color: #9cdcfe;">T</span></sup>) 
             2.1.2.1 <span style="color: #9cdcfe;">candidata</span> ← <span style="color: #dcdcaa;">genera_sucesor</span>(<span style="color: #9cdcfe;">actual</span>, <span style="color: #9cdcfe;">i</span>, <span style="color: #9cdcfe;">j</span>)                
                     <span style="color: #9cdcfe;">valor_candidata</span> ← <span style="color: #dcdcaa;">valoracion</span>(<span style="color: #9cdcfe;">candidata</span>, <span style="color: #9cdcfe;">W</span>)            
                     <span style="color: #9cdcfe;">ΔF</span> ← <span style="color: #9cdcfe;">valor_candidata</span> - <span style="color: #9cdcfe;">valor_actual</span>
             2.1.2.2 <b style="color: #c586c0;">Si</b> <span style="color: #9cdcfe;">ΔF</span> &lt; <span style="color: #b5cea8;">0</span> <b style="color: #c586c0;">ó</b>
                        <b style="color: #c586c0;">se acepta</b> (con probabilidad de aceptación <span style="color: #dcdcaa;">e</span><sup>-<span style="color: #9cdcfe;">ΔF</span>/<span style="color: #9cdcfe;">T</span></sup>)   
                     2.1.2.2.1 <span style="color: #9cdcfe;">actual</span> ← <span style="color: #9cdcfe;">candidata</span>
                               <span style="color: #9cdcfe;">valor_actual</span> ← <span style="color: #9cdcfe;">valor_candidata</span>
       2.1.3 <b style="color: #c586c0;">Si</b> <span style="color: #9cdcfe;">valor_actual</span> &lt; <span style="color: #9cdcfe;">mejor_valor</span>
             2.1.3.1 <span style="color: #9cdcfe;">mejor</span> ← <span style="color: #9cdcfe;">actual</span>
                     <span style="color: #9cdcfe;">mejor_valor</span> ← <span style="color: #9cdcfe;">valor_actual</span>
   2.2 <span style="color: #9cdcfe;">T</span> ← <span style="color: #9cdcfe;">α</span> · <span style="color: #9cdcfe;">T</span>

3. <span style="color: #9cdcfe;">mejor_rotada</span> ← <span style="color: #dcdcaa;">rotar_ruta_al_nodo</span>(<span style="color: #9cdcfe;">mejor</span>, <span style="color: #9cdcfe;">inicio</span>)
4. <b style="color: #c586c0;">Devolver</b> <span style="color: #9cdcfe;">mejor_rotada</span> y <span style="color: #9cdcfe;">mejor_valor</span>
</pre>

**💻 CÓDIGO**

`valoracion` calcula el coste de una ruta, `genera_sucesor` aplica el movimiento 2-opt (invertir un tramo), `enfriamiento_simulado` es el bucle principal (acepta empeoramientos con probabilidad $e^{-\Delta F/T}$ mientras baja la temperatura) y `estimar_cota_1_arbol` calcula una cota inferior del óptimo para acotar el error.

In [ ]:
import random
import math
import networkx as nx

def valoracion(ruta, W, N):
    c = 0.0
    for k in range(N):
        c += W.get((ruta[k-1], ruta[k]), float('inf'))
    return c

def genera_sucesor(actual, i, j):
    """Genera una nueva ruta aplicando el operador 2-opt (inversión de subcadena)."""
    return actual[:i] + actual[i:j+1][::-1] + actual[j+1:]

def enfriamiento_simulado(K, inicio=None):
    nodos = list(K.nodes)
    N = len(nodos)
    if N <= 1:
        return nodos, 0.0

    if inicio not in nodos:
        inicio = nodos[0]
    
    # 1. Caché de distancias O(1)
    W = {(u, v): data["peso"] for u, v, data in K.edges(data=True)}
    if not K.is_directed():                  # grafo no dirigido: cada arista se guarda en los dos sentidos
        W.update({(v, u): peso for (u, v), peso in list(W.items())})
    
    # 2. Estado inicial aleatorio
    actual = nodos[:]
    random.shuffle(actual)
    
    mejor = actual[:]
    valor_actual = valoracion(actual, W, N)
    mejor_valor = valor_actual

    # 3. Parámetros Termodinámicos
    T_inicial = valor_actual * 0.15      
    T = T_inicial
    T_min = 0.05                         
    alpha = 0.95

    while T > T_min:
        factor_frio = T_inicial / T
        n_iter = int(N * 5 * min(3.0, factor_frio))

        for _ in range(n_iter):
            i = random.randrange(N)
            j = random.randrange(N)
            if i == j or j - i == N - 1:
                continue
            if i > j:
                i, j = j, i

            u_prev, u = actual[i-1], actual[i]
            v, v_next = actual[j], actual[(j+1) % N]

            coste_viejas = W.get((u_prev, u), float('inf')) + W.get((v, v_next), float('inf'))
            coste_nuevas = W.get((u_prev, v), float('inf')) + W.get((u, v_next), float('inf'))
            
            delta_F_est = coste_nuevas - coste_viejas

            if delta_F_est <= 0 or random.random() < math.exp(-delta_F_est / T):
                candidata = genera_sucesor(actual, i, j)
                valor_candidata = valoracion(candidata, W, N)
                delta_F = valor_candidata - valor_actual

                if delta_F <= 0 or random.random() < math.exp(-delta_F / T):
                    actual = candidata
                    valor_actual = valor_candidata
                    
                    if valor_actual < mejor_valor:
                        mejor_valor = valor_actual
                        mejor = actual[:]

        T *= alpha 

    idx = mejor.index(inicio)
    mejor_rotada = mejor[idx:] + mejor[:idx]

    return mejor_rotada, mejor_valor


def estimar_cota_1_arbol(K):
    """
    Calcula una cota inferior estricta usando un 1-Árbol.
    """
    nodos = list(K.nodes)
    if len(nodos) < 3:
        return 0.0
        
    nodo_base = nodos[0]
    nodos_restantes = list(nodos[1:])
    
    visitados = {nodos_restantes[0]}
    coste_mst = 0.0
    obtener_vecinos = K.successors if isinstance(K, nx.DiGraph) else K.neighbors
    
    while len(visitados) < len(nodos_restantes):
        min_peso = float('inf')
        mejor_nodo = None
        for u in visitados:
            for v in obtener_vecinos(u):
                if v in nodos_restantes and v not in visitados:
                    peso = K[u][v]["peso"]
                    if peso < min_peso:
                        min_peso = peso
                        mejor_nodo = v
        if mejor_nodo:
            visitados.add(mejor_nodo)
            coste_mst += min_peso
        else:
            break
            
    if isinstance(K, nx.DiGraph):
        min_salida = min([K[nodo_base][v]["peso"] for v in nodos_restantes if K.has_edge(nodo_base, v)], default=0)
        min_entrada = min([K[v][nodo_base]["peso"] for v in nodos_restantes if K.has_edge(v, nodo_base)], default=0)
        coste_conexiones = min_salida + min_entrada
    else:
        pesos_base = sorted([K[nodo_base][v]["peso"] for v in nodos_restantes if K.has_edge(nodo_base, v)])
        coste_conexiones = sum(pesos_base[:2]) if len(pesos_base) >= 2 else 0

    return coste_mst + coste_conexiones

**🧪 PRUEBAS**

In [9]:
random.seed(42)    # semilla fija: así las pruebas de abajo se pueden repetir

In [10]:
# 1. Prueba rápida en Galicia (con impresión de ruta y rendimiento completo)
K = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="galicia", peso="km", general="grafo_dirigido")
ruta, coste = enfriamiento_simulado(K, inicio="Vigo")
validar_ruta(K, ruta)
print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
coste_mst = estimar_cota_1_arbol(K)
error_max = ((coste - coste_mst) / coste_mst) * 100 if coste_mst > 0 else 0
print(f"Error Máx < {error_max:.2f}% (Cota inferior MST: {coste_mst:.1f} {K.graph['unidad']})")
# ----------------------------
print("################Rendimiento####################")
print(medir(enfriamiento_simulado, K, memoria=True, inicio="Vigo"))


Vigo → Ourense → Lugo → A Coruña → Santiago de Compostela → Pontevedra | coste: 460.4 km
Error Máx < 7.78% (Cota inferior MST: 427.2 km)
################Rendimiento####################
{'algoritmo': 'enfriamiento_simulado', 'ciudades': 6, 'coste': 460.438, 'unidad': 'km', 'segundos': 0.063, 'memoria_mb': 0.0}


In [11]:
# 2. Prueba con Capitales de España (Sevilla)
K = cargar_version(aristas="completo", nodos="capitales", ambito="espana", peso="km", general="grafo_dirigido")
ruta, coste = enfriamiento_simulado(K, inicio="Sevilla")
validar_ruta(K, ruta)
print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
coste_mst = estimar_cota_1_arbol(K)
error_max = ((coste - coste_mst) / coste_mst) * 100 if coste_mst > 0 else 0
print(f"Error Máx < {error_max:.2f}% (Cota inferior MST: {coste_mst:.1f} {K.graph['unidad']})")
# ----------------------------
print("################Rendimiento####################")
print(medir(enfriamiento_simulado, K, memoria=True, inicio="Sevilla"))

Sevilla → Huelva → Badajoz → Cáceres → Ávila → Salamanca → Zamora → Ourense → Pontevedra → A Coruña → Lugo → León → Oviedo → Santander → Bilbao → Vitoria-Gasteiz → San Sebastián → Pamplona → Zaragoza → Huesca → Lleida → Girona → Barcelona → Tarragona → Castellón de la Plana → Valencia → Alicante → Murcia → Albacete → Cuenca → Teruel → Soria → Logroño → Burgos → Palencia → Valladolid → Segovia → Guadalajara → Madrid → Toledo → Ciudad Real → Córdoba → Jaén → Almería → Granada → Málaga → Cádiz | coste: 6160.2 km
Error Máx < 21.45% (Cota inferior MST: 5072.3 km)
################Rendimiento####################


{'algoritmo': 'enfriamiento_simulado', 'ciudades': 47, 'coste': 6092.996, 'unidad': 'km', 'segundos': 0.8388, 'memoria_mb': 0.11}


In [12]:

# 3. Prueba con Capitales y +100k de España (Vigo - Completo)
K = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="espana", peso="km", general="grafo_dirigido")
ruta, coste = enfriamiento_simulado(K, inicio="Vigo")
validar_ruta(K, ruta)
print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
coste_mst = estimar_cota_1_arbol(K)
error_max = ((coste - coste_mst) / coste_mst) * 100 if coste_mst > 0 else 0
print(f"Error Máx < {error_max:.2f}% (Cota inferior MST: {coste_mst:.1f} {K.graph['unidad']})")
# ----------------------------
print("################Rendimiento####################")
print(medir(enfriamiento_simulado, K, memoria=True, inicio="Vigo"))


Vigo → Ourense → León → Zamora → Valladolid → Palencia → Burgos → Segovia → Ávila → Salamanca → Cáceres → Badajoz → Huelva → Sevilla → Dos Hermanas → Jerez de la Frontera → Cádiz → Algeciras → Marbella → Málaga → Roquetas de Mar → Almería → Granada → Jaén → Córdoba → Ciudad Real → Toledo → Parla → Fuenlabrada → Getafe → Leganés → Alcorcón → Móstoles → Madrid → Alcobendas → Rivas-Vaciamadrid → Torrejón de Ardoz → Alcalá de Henares → Guadalajara → Cuenca → Teruel → Albacete → Murcia → Cartagena → Elche → Alicante → Valencia → Castellón de la Plana → Reus → Tarragona → L'Hospitalet de Llobregat → Barcelona → Santa Coloma de Gramenet → Badalona → Mataró → Girona → Sabadell → Terrassa → Lleida → Huesca → Zaragoza → Soria → Logroño → Pamplona → San Sebastián → Vitoria-Gasteiz → Bilbao → Barakaldo → Santander → Gijón → Oviedo → Lugo → A Coruña → Santiago de Compostela → Pontevedra | coste: 6379.1 km
Error Máx < 19.61% (Cota inferior MST: 5333.2 km)
################Rendimiento#################

{'algoritmo': 'enfriamiento_simulado', 'ciudades': 75, 'coste': 6408.542, 'unidad': 'km', 'segundos': 2.125, 'memoria_mb': 0.64}


In [13]:


# 4. Prueba con Autovías Principales
K = cargar_version(aristas="autovias_principales", nodos="capitales_y_100k", ambito="espana", peso="km", general="grafo_dirigido")
ruta, coste = enfriamiento_simulado(K, inicio="Vigo")
validar_ruta(K, ruta)
print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
coste_mst = estimar_cota_1_arbol(K)
error_max = ((coste - coste_mst) / coste_mst) * 100 if coste_mst > 0 else 0
print(f"Error Máx < {error_max:.2f}% (Cota inferior MST: {coste_mst:.1f} {K.graph['unidad']})")
# ----------------------------
print("################Rendimiento####################")
print(medir(enfriamiento_simulado, K, memoria=True, inicio="Vigo"))



Vigo → Pontevedra → Santiago de Compostela → A Coruña → Lugo → Oviedo → Gijón → Santander → Barakaldo → Bilbao → Vitoria-Gasteiz → San Sebastián → Pamplona → Logroño → Soria → Zaragoza → Huesca → Lleida → Terrassa → Sabadell → Santa Coloma de Gramenet → Mataró → Girona → Badalona → Barcelona → L'Hospitalet de Llobregat → Tarragona → Reus → Castellón de la Plana → Valencia → Alicante → Elche → Cartagena → Murcia → Albacete → Teruel → Cuenca → Guadalajara → Alcalá de Henares → Torrejón de Ardoz → Rivas-Vaciamadrid → Madrid → Alcobendas → Segovia → Móstoles → Alcorcón → Leganés → Getafe → Parla → Fuenlabrada → Toledo → Ciudad Real → Córdoba → Jaén → Granada → Roquetas de Mar → Almería → Málaga → Marbella → Algeciras → Cádiz → Jerez de la Frontera → Dos Hermanas → Sevilla → Huelva → Badajoz → Cáceres → Ávila → Salamanca → Zamora → Valladolid → Palencia → Burgos → León → Ourense | coste: 6674.7 km
Error Máx < 20.22% (Cota inferior MST: 5551.8 km)
################Rendimiento#################

{'algoritmo': 'enfriamiento_simulado', 'ciudades': 75, 'coste': 6582.988, 'unidad': 'km', 'segundos': 2.5685, 'memoria_mb': 0.64}


In [14]:
import statistics

# 5. Prueba con Solo Autovías (Análisis de Desviación y Estabilidad)
K = cargar_version(aristas="solo_autovias", nodos="capitales_y_100k", ambito="espana", peso="km", general="grafo_dirigido")
inicio_ruta = "Vigo"
num_pruebas = 50  # Número de veces que ejecutaremos el algoritmo para medir su estabilidad

costes_obtenidos = []
mejor_ruta_global = None
mejor_coste_global = float('inf')

print(f"Ejecutando {num_pruebas} pruebas de recocido simulado para calcular la desviación...")

for i in range(num_pruebas):
    ruta, coste = enfriamiento_simulado(K, inicio=inicio_ruta)
    costes_obtenidos.append(coste)
    
    if coste < mejor_coste_global:
        mejor_coste_global = coste
        mejor_ruta_global = ruta

# Cálculos estadísticos
coste_medio = statistics.mean(costes_obtenidos)
desviacion_estandar = statistics.stdev(costes_obtenidos)
peor_coste = max(costes_obtenidos)

# El coeficiente de variación mide el % de inestabilidad relativa
coef_variacion = (desviacion_estandar / coste_medio) * 100 

# Validar solo la mejor ruta obtenida
validar_ruta(K, mejor_ruta_global)

# Calcular el error respecto a la cota 1-Árbol
coste_cota = estimar_cota_1_arbol(K)
error_mejor = ((mejor_coste_global - coste_cota) / coste_cota) * 100 if coste_cota > 0 else 0
error_medio = ((coste_medio - coste_cota) / coste_cota) * 100 if coste_cota > 0 else 0
unidad = K.graph.get('unidad', 'km')

# Imprimir el informe de estabilidad
print("\n" + "="*50)
print("📊 INFORME DE DESVIACIÓN (METAHEURÍSTICA ESTOCÁSTICA)")
print("="*50)
print(f"• Pruebas realizadas : {num_pruebas}")
print(f"• Cota 1-Árbol base  : {coste_cota:.1f} {unidad}")
print("-" * 50)
print(f"• MEJOR coste        : {mejor_coste_global:.1f} {unidad} (Error < {error_mejor:.2f}%)")
print(f"• PEOR coste         : {peor_coste:.1f} {unidad}")
print(f"• Coste MEDIO        : {coste_medio:.1f} {unidad} (Error < {error_medio:.2f}%)")
print("-" * 50)
print(f"• Desviación Estándar: ±{desviacion_estandar:.1f} {unidad}")
print(f"• Coef. de Variación : {coef_variacion:.2f}% (A menor %, mayor fiabilidad)")
print("="*50)

print(f"\nMejor ruta encontrada:\n{' → '.join(mejor_ruta_global)}")

Ejecutando 50 pruebas de recocido simulado para calcular la desviación...



📊 INFORME DE DESVIACIÓN (METAHEURÍSTICA ESTOCÁSTICA)
• Pruebas realizadas : 50
• Cota 1-Árbol base  : 7270.4 km
--------------------------------------------------
• MEJOR coste        : 8826.6 km (Error < 21.40%)
• PEOR coste         : 11256.6 km
• Coste MEDIO        : 9707.3 km (Error < 33.52%)
--------------------------------------------------
• Desviación Estándar: ±475.0 km
• Coef. de Variación : 4.89% (A menor %, mayor fiabilidad)

Mejor ruta encontrada:
Vigo → Pontevedra → Santiago de Compostela → A Coruña → Lugo → León → Zamora → Valladolid → Palencia → Burgos → Alcobendas → Segovia → Ávila → Rivas-Vaciamadrid → Torrejón de Ardoz → Alcalá de Henares → Soria → Guadalajara → Móstoles → Madrid → Parla → Getafe → Alcorcón → Leganés → Fuenlabrada → Toledo → Ciudad Real → Cáceres → Sevilla → Huelva → Dos Hermanas → Jerez de la Frontera → Cádiz → Algeciras → Marbella → Córdoba → Granada → Jaén → Málaga → Roquetas de Mar → Almería → Cartagena → Murcia → Elche → Alicante → Albacete → Va

In [15]:

# 5. Prueba con Solo Autovías
K = cargar_version(aristas="solo_autovias", nodos="capitales_y_100k", ambito="espana", peso="km", general="grafo_dirigido")
ruta, coste = enfriamiento_simulado(K, inicio="Vigo")
validar_ruta(K, ruta)
print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
coste_mst = estimar_cota_1_arbol(K)
error_max = ((coste - coste_mst) / coste_mst) * 100 if coste_mst > 0 else 0
print(f"Error Máx < {error_max:.2f}% (Cota inferior MST: {coste_mst:.1f} {K.graph['unidad']})")
# ----------------------------
# print("################Rendimiento####################")
# print(medir(enfriamiento_simulado, K, memoria=True, inicio="Vigo"))





Vigo → Pontevedra → Santiago de Compostela → A Coruña → Lugo → León → Zamora → Getafe → Ciudad Real → Albacete → Alcobendas → Burgos → Palencia → Valladolid → Segovia → Madrid → Torrejón de Ardoz → Alcalá de Henares → Soria → Guadalajara → Rivas-Vaciamadrid → Móstoles → Ávila → Parla → Alcorcón → Leganés → Fuenlabrada → Toledo → Cáceres → Sevilla → Huelva → Dos Hermanas → Córdoba → Jerez de la Frontera → Cádiz → Algeciras → Marbella → Jaén → Málaga → Granada → Roquetas de Mar → Almería → Cartagena → Murcia → Elche → Alicante → Valencia → Teruel → Castellón de la Plana → Tarragona → Reus → Girona → Santa Coloma de Gramenet → Badalona → Mataró → Sabadell → Terrassa → L'Hospitalet de Llobregat → Barcelona → Lleida → Huesca → Zaragoza → Logroño → Vitoria-Gasteiz → Pamplona → San Sebastián → Barakaldo → Bilbao → Santander → Gijón → Oviedo → Ourense | coste: 9485.2 km
Error Máx < 30.46% (Cota inferior MST: 7270.4 km)


In [16]:

# 6. Prueba Sin Autovías
K = cargar_version(aristas="sin_autovias", nodos="capitales_y_100k", ambito="espana", peso="km", general="grafo_dirigido")
ruta, coste = enfriamiento_simulado(K, inicio="Vigo")
validar_ruta(K, ruta)
print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
coste_mst = estimar_cota_1_arbol(K)
error_max = ((coste - coste_mst) / coste_mst) * 100 if coste_mst > 0 else 0
print(f"Error Máx < {error_max:.2f}% (Cota inferior MST: {coste_mst:.1f} {K.graph['unidad']})")
# ----------------------------
# print("################Rendimiento####################")
# print(medir(enfriamiento_simulado, K, memoria=True, inicio="Vigo"))


Vigo → Pontevedra → Santiago de Compostela → A Coruña → Lugo → Oviedo → Gijón → Santander → Barakaldo → Bilbao → San Sebastián → Pamplona → Huesca → Zaragoza → Lleida → Barcelona → Santa Coloma de Gramenet → L'Hospitalet de Llobregat → Terrassa → Sabadell → Girona → Mataró → Badalona → Tarragona → Reus → Castellón de la Plana → Valencia → Teruel → Cuenca → Alcalá de Henares → Torrejón de Ardoz → Rivas-Vaciamadrid → Madrid → Leganés → Móstoles → Parla → Toledo → Ciudad Real → Albacete → Elche → Alicante → Cartagena → Murcia → Jaén → Granada → Almería → Roquetas de Mar → Málaga → Marbella → Algeciras → Jerez de la Frontera → Cádiz → Córdoba → Dos Hermanas → Sevilla → Huelva → Badajoz → Cáceres → Salamanca → Zamora → Valladolid → Segovia → Ávila → Alcorcón → Fuenlabrada → Getafe → Alcobendas → Guadalajara → Soria → Logroño → Vitoria-Gasteiz → Burgos → Palencia → León → Ourense | coste: 8758.2 km
Error Máx < 22.83% (Cota inferior MST: 7130.3 km)


In [17]:


# 7. Pruebas de variabilidad con distintos inicios (Madrid, Barcelona, etc.)
for ciudad_inicio in ["Sevilla", "Madrid", "Barcelona", "Pontevedra"]:
    K = cargar_version(aristas="completo", nodos="capitales", ambito="espana", peso="km", general="grafo_dirigido")
    ruta, coste = enfriamiento_simulado(K, inicio=ciudad_inicio)
    validar_ruta(K, ruta)
    print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
    coste_mst = estimar_cota_1_arbol(K)
    error_max = ((coste - coste_mst) / coste_mst) * 100 if coste_mst > 0 else 0
    print(f"Error Máx < {error_max:.2f}% (Cota inferior MST: {coste_mst:.1f} {K.graph['unidad']})")
    # ----------------------------
    print(medir(enfriamiento_simulado, K, memoria=True, inicio=ciudad_inicio))
    print("################Rendimiento####################")



Sevilla → Huelva → Badajoz → Cáceres → Salamanca → Ávila → Segovia → Valladolid → Palencia → Burgos → León → Zamora → Ourense → Pontevedra → A Coruña → Lugo → Oviedo → Santander → Bilbao → San Sebastián → Pamplona → Vitoria-Gasteiz → Logroño → Soria → Guadalajara → Madrid → Toledo → Ciudad Real → Albacete → Cuenca → Teruel → Zaragoza → Huesca → Lleida → Girona → Barcelona → Tarragona → Castellón de la Plana → Valencia → Alicante → Murcia → Almería → Granada → Jaén → Córdoba → Málaga → Cádiz | coste: 6092.7 km
Error Máx < 20.12% (Cota inferior MST: 5072.3 km)


{'algoritmo': 'enfriamiento_simulado', 'ciudades': 47, 'coste': 6176.738, 'unidad': 'km', 'segundos': 0.7968, 'memoria_mb': 0.11}
################Rendimiento####################


Madrid → Segovia → Ávila → Toledo → Ciudad Real → Córdoba → Jaén → Granada → Almería → Málaga → Cádiz → Sevilla → Huelva → Badajoz → Cáceres → Salamanca → Zamora → Ourense → Pontevedra → A Coruña → Lugo → Oviedo → León → Valladolid → Palencia → Burgos → Santander → Bilbao → San Sebastián → Pamplona → Vitoria-Gasteiz → Logroño → Soria → Zaragoza → Huesca → Lleida → Girona → Barcelona → Tarragona → Castellón de la Plana → Teruel → Valencia → Alicante → Murcia → Albacete → Cuenca → Guadalajara | coste: 5982.3 km
Error Máx < 17.94% (Cota inferior MST: 5072.3 km)


{'algoritmo': 'enfriamiento_simulado', 'ciudades': 47, 'coste': 6011.936, 'unidad': 'km', 'segundos': 0.8552, 'memoria_mb': 0.11}
################Rendimiento####################


Barcelona → Girona → Lleida → Huesca → Zaragoza → Pamplona → San Sebastián → Bilbao → Santander → Burgos → Vitoria-Gasteiz → Logroño → Soria → Segovia → Ávila → Valladolid → Palencia → León → Oviedo → Lugo → A Coruña → Pontevedra → Ourense → Zamora → Salamanca → Cáceres → Badajoz → Huelva → Sevilla → Cádiz → Córdoba → Jaén → Granada → Málaga → Almería → Murcia → Alicante → Albacete → Ciudad Real → Toledo → Madrid → Guadalajara → Cuenca → Teruel → Valencia → Castellón de la Plana → Tarragona | coste: 6083.2 km
Error Máx < 19.93% (Cota inferior MST: 5072.3 km)


{'algoritmo': 'enfriamiento_simulado', 'ciudades': 47, 'coste': 6110.001, 'unidad': 'km', 'segundos': 0.8325, 'memoria_mb': 0.11}
################Rendimiento####################


Pontevedra → A Coruña → Lugo → Oviedo → León → Valladolid → Palencia → Burgos → Santander → Bilbao → San Sebastián → Pamplona → Vitoria-Gasteiz → Logroño → Soria → Zaragoza → Huesca → Lleida → Girona → Barcelona → Tarragona → Castellón de la Plana → Teruel → Cuenca → Valencia → Alicante → Murcia → Albacete → Ciudad Real → Jaén → Granada → Almería → Málaga → Córdoba → Cádiz → Sevilla → Huelva → Badajoz → Cáceres → Toledo → Madrid → Guadalajara → Segovia → Ávila → Salamanca → Zamora → Ourense | coste: 6170.6 km
Error Máx < 21.65% (Cota inferior MST: 5072.3 km)


{'algoritmo': 'enfriamiento_simulado', 'ciudades': 47, 'coste': 6040.594, 'unidad': 'km', 'segundos': 0.8852, 'memoria_mb': 0.11}
################Rendimiento####################


In [18]:

# 8. Prueba en ámbito Andalucía
K = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="andalucia", peso="km", general="grafo_dirigido")
ruta, coste = enfriamiento_simulado(K, inicio="Sevilla")
validar_ruta(K, ruta)
coste_mst = estimar_cota_1_arbol(K)
error_max = ((coste - coste_mst) / coste_mst) * 100 if coste_mst > 0 else 0
print(f"Error Máx < {error_max:.2f}% (Cota inferior MST: {coste_mst:.1f} {K.graph['unidad']})")
# ----------------------------
print("################Rendimiento####################")
print(medir(enfriamiento_simulado, K, memoria=True, inicio="Sevilla"))

Error Máx < 27.37% (Cota inferior MST: 982.4 km)
################Rendimiento####################


{'algoritmo': 'enfriamiento_simulado', 'ciudades': 13, 'coste': 1251.307, 'unidad': 'km', 'segundos': 0.0958, 'memoria_mb': 0.01}


<div style="border-left:5px solid #55A868;color:#000;background:#EAF6ED;padding:10px 14px;margin:10px 0;border-radius:4px;">

**🧮 RESULTADOS**

Medidos con `random.seed(42)`, versión `completo`/`km`/dirigido salvo que se indique. «Error máx.» es el error respecto a la cota inferior del 1-Árbol (el error real es menor). El óptimo de las 47 capitales es 5.917,6 km (DFJ).

| Nº ciudades | Versión | Coste (km) | Tiempo (s) | Memoria (MB) | Error máx. (1-Árbol) |
| :---: | :---: | :---: | :---: | :---: | :---: |
| 6 | Galicia | 460,4 (óptimo) | 0,06 | < 0,1 | 7,8 % |
| 13 | Andalucía | 1.251,3 (óptimo) | 0,10 | < 0,1 | 27,4 % |
| 47 | capitales (desde Sevilla) | 6.093,0 – 6.176,7 | 0,8 | 0,11 | 20,1 – 21,5 % |
| 47 | capitales (4 inicios: Sevilla, Madrid, Barcelona, Pontevedra) | 5.982,3 – 6.176,7 | 0,8 – 0,9 | 0,11 | 17,9 – 21,7 % |
| 75 | capitales y > 100.000 hab. | 6.408,5 | 2,1 | 0,64 | 19,6 % |
| 75 | autovías principales | 6.583,0 | 2,6 | 0,64 | 20,2 % |

**Estabilidad** (75 ciudades, solo autovías, 50 ejecuciones): mejor 8.826,6 km, peor 11.256,6 km, media 9.707,3 km, desviación ±475,0 km (coeficiente de variación 4,9 %).

Con 47 capitales queda entre un **+1,1 % y un +4,4 %** sobre el óptimo, según el arranque. El tiempo crece de forma suave con el número de ciudades (≈ 1 s con 47 y 2–3 s con 75) y la memoria se mantiene en menos de 1 MB.

*Nota: a diferencia de los algoritmos exactos, el recorrido que devuelve depende del azar y del arranque; por eso se indican rangos y se mide la estabilidad sobre muchas ejecuciones.*
</div>

---
